In [1]:
import os
from dotenv import load_dotenv

import duckdb

load_dotenv()

True

In [2]:
con = duckdb.connect('md:')

In [3]:
#get two pages

import random
from time import sleep

import undetected_chromedriver as uc

from scrape.utils import filter_html
from scrape.utils import get_listings
from scrape.utils import get_page_count
from scrape.utils import make_page_urls
from scrape.constants import FREGUESIA_URL

driver = uc.Chrome()
driver.get("https://www.idealista.pt/arrendar-casas/lisboa/ajuda/")
sleep(6)
raw_html_ajuda = driver.page_source


sleep(3)
driver.get("https://www.idealista.pt/arrendar-casas/lisboa/alcantara/")
raw_html_alcantara = driver.page_source

listings_ajuda, status1 = get_listings(raw_html_ajuda)
listings_alcantara, status2 = get_listings(raw_html_alcantara)


In [4]:
a= filter_html(listings_ajuda,"ajuda")

In [9]:
type(a[0])

dict

In [12]:
con.sql("""
    CREATE TABLE IF NOT EXISTS raw_listings (
        id VARCHAR,
        price VARCHAR,
        title VARCHAR,
        extra VARCHAR[],
        freguesia VARCHAR,
        scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

In [ ]:
con.sql("USE raw_listings")

In [13]:
con.executemany(
    "INSERT INTO raw_listings (id, price, title, extra, freguesia) VALUES (?, ?, ?, ?, ?)",
    [[d["id"], d["price"], d["title"], d["extra"], d["freguesia"]] for d in a]
)